
# Roxy notebook example: Run and blockiness descriptors

This notebook is a **reference implementation example** for the **run and blockiness descriptor family** in Roxy.

Run and blockiness descriptors try to capture whether certain residues or residue groups appear in **compact consecutive blocks**, **fragmented patterns**, or **long uninterrupted stretches** along the sequence.

## Covered outputs

This notebook implements examples such as:

- longest run length
- number of runs
- mean run length
- normalized longest run
- run density
- fraction of residues inside runs of length >= threshold
- longest homopolymer run
- longest group-specific run
- blockiness score
- switching frequency between grouped states
- class-style implementation for later migration into Roxy

These descriptors are useful because two sequences can have similar composition but very different **local organization**.


In [1]:

from itertools import groupby

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "run_1",
            "run_2",
            "run_3",
            "run_4",
            "run_5",
            "run_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,run_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,run_2,GGGGGGGGGGGGGGG,B
2,run_3,KRRKRRKRRKRRDDDDEE,A
3,run_4,ACDEFGHIKLMNPQRSTVWY,B
4,run_5,PPPPGSSSSSTTTTNNQQQ,A
5,run_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

AA_GROUPS = {
    "positive": set("KRH"),
    "negative": set("DE"),
    "charged": set("KRHDE"),
    "polar": set("STNQCYWHKRDE"),
    "nonpolar": set("AVLIMFGP"),
    "aromatic": set("FWYH"),
    "aliphatic": set("AVLIM"),
    "hydrophobic": set("AVLIMFWCY"),
    "hydrophilic": set("RNDQEHKST"),
    "disorder_promoting": set("ARGQSEPK"),
    "order_promoting": set("CWYFILNV"),
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def run_lengths_from_binary(binary_vector):
    runs = [len(list(group)) for value, group in groupby(binary_vector) if value == 1]
    return runs


def binary_membership(seq: str, aa_group):
    return [1 if aa in aa_group else 0 for aa in seq]


def longest_run(seq: str, aa_group) -> int:
    runs = run_lengths_from_binary(binary_membership(seq, aa_group))
    return max(runs) if runs else 0


def number_of_runs(seq: str, aa_group) -> int:
    runs = run_lengths_from_binary(binary_membership(seq, aa_group))
    return len(runs)


def mean_run_length(seq: str, aa_group) -> float:
    runs = run_lengths_from_binary(binary_membership(seq, aa_group))
    if not runs:
        return np.nan
    return float(np.mean(runs))


def normalized_longest_run(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return longest_run(seq, aa_group) / len(seq)


def run_density(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return number_of_runs(seq, aa_group) / len(seq)


def fraction_in_long_runs(seq: str, aa_group, min_run_length: int = 2) -> float:
    if len(seq) == 0:
        return np.nan
    runs = run_lengths_from_binary(binary_membership(seq, aa_group))
    if not runs:
        return 0.0
    residues_in_long_runs = sum(run for run in runs if run >= min_run_length)
    return residues_in_long_runs / len(seq)


def blockiness_score(seq: str, aa_group) -> float:
    """Simple blockiness proxy: mean run length scaled by number of matching residues."""
    binary = binary_membership(seq, aa_group)
    total_hits = sum(binary)
    if len(seq) == 0 or total_hits == 0:
        return np.nan
    runs = run_lengths_from_binary(binary)
    if not runs:
        return np.nan
    return float(np.mean(runs) / total_hits)


def switching_frequency(seq: str, aa_group) -> float:
    """Frequency of switching between in-group and out-group along the sequence."""
    if len(seq) < 2:
        return np.nan
    binary = binary_membership(seq, aa_group)
    switches = sum(binary[i] != binary[i + 1] for i in range(len(binary) - 1))
    return switches / (len(binary) - 1)


def longest_homopolymer_run(seq: str) -> int:
    if len(seq) == 0:
        return 0
    return max(len(list(group)) for _, group in groupby(seq))


def homopolymer_run_density(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    homo_runs = [len(list(group)) for _, group in groupby(seq)]
    return len(homo_runs) / len(seq)


def repeated_block_fraction(seq: str, min_run_length: int = 2) -> float:
    if len(seq) == 0:
        return np.nan
    homo_runs = [len(list(group)) for _, group in groupby(seq)]
    residues_in_blocks = sum(run for run in homo_runs if run >= min_run_length)
    return residues_in_blocks / len(seq)


## Core descriptor function

In [5]:

def run_blockiness_descriptors(seq: str) -> dict:
    seq = clean_sequence(seq)

    out = {
        "run_length": len(seq),
        "run_valid_residue_count": len(seq),
        "run_longest_homopolymer": longest_homopolymer_run(seq),
        "run_homopolymer_run_density": homopolymer_run_density(seq),
        "run_repeated_block_fraction_len2": repeated_block_fraction(seq, min_run_length=2),
        "run_repeated_block_fraction_len3": repeated_block_fraction(seq, min_run_length=3),
    }

    tracked_groups = [
        "charged",
        "hydrophobic",
        "polar",
        "aromatic",
        "disorder_promoting",
        "order_promoting",
    ]

    for name in tracked_groups:
        group = AA_GROUPS[name]
        out[f"run_{name}_longest"] = longest_run(seq, group)
        out[f"run_{name}_n_runs"] = number_of_runs(seq, group)
        out[f"run_{name}_mean_run_length"] = mean_run_length(seq, group)
        out[f"run_{name}_longest_norm"] = normalized_longest_run(seq, group)
        out[f"run_{name}_run_density"] = run_density(seq, group)
        out[f"run_{name}_fraction_in_runs_len2"] = fraction_in_long_runs(seq, group, min_run_length=2)
        out[f"run_{name}_fraction_in_runs_len3"] = fraction_in_long_runs(seq, group, min_run_length=3)
        out[f"run_{name}_blockiness"] = blockiness_score(seq, group)
        out[f"run_{name}_switching_freq"] = switching_frequency(seq, group)

    return out


## Functional usage on one sequence

In [6]:

example = run_blockiness_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:18]


[('run_length', 24),
 ('run_valid_residue_count', 24),
 ('run_longest_homopolymer', 2),
 ('run_homopolymer_run_density', 0.875),
 ('run_repeated_block_fraction_len2', 0.25),
 ('run_repeated_block_fraction_len3', 0.0),
 ('run_charged_longest', 2),
 ('run_charged_n_runs', 3),
 ('run_charged_mean_run_length', 1.3333333333333333),
 ('run_charged_longest_norm', 0.08333333333333333),
 ('run_charged_run_density', 0.125),
 ('run_charged_fraction_in_runs_len2', 0.08333333333333333),
 ('run_charged_fraction_in_runs_len3', 0.0),
 ('run_charged_blockiness', 0.3333333333333333),
 ('run_charged_switching_freq', 0.21739130434782608),
 ('run_hydrophobic_longest', 5),
 ('run_hydrophobic_n_runs', 6),
 ('run_hydrophobic_mean_run_length', 2.3333333333333335)]

## Apply run and blockiness descriptors to the full dataset

In [7]:

df_run = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(run_blockiness_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_run.head()


,sequence_id,sequence,label,run_length,run_valid_residue_count,run_longest_homopolymer,run_homopolymer_run_density,run_repeated_block_fraction_len2,run_repeated_block_fraction_len3,run_charged_longest,...,run_disorder_promoting_switching_freq,run_order_promoting_longest,run_order_promoting_n_runs,run_order_promoting_mean_run_length,run_order_promoting_longest_norm,run_order_promoting_run_density,run_order_promoting_fraction_in_runs_len2,run_order_promoting_fraction_in_runs_len3,run_order_promoting_blockiness,run_order_promoting_switching_freq
0,run_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,2.0,0.875000,0.250000,0.000000,2.0,...,0.391304,5.0,5.0,2.400000,0.208333,0.208333,0.458333,0.208333,0.200000,0.434783
1,run_2,GGGGGGGGGGGGGGG,B,15.0,15.0,15.0,0.066667,1.000000,1.000000,0.0,...,0.000000,0.0,0.0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
2,run_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,4.0,0.555556,0.777778,0.222222,18.0,...,0.117647,0.0,0.0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
3,run_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,1.0,1.000000,0.000000,0.000000,2.0,...,0.473684,3.0,6.0,1.333333,0.150000,0.300000,0.150000,0.150000,0.166667,0.578947
4,run_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,5.0,0.315789,0.947368,0.842105,0.0,...,0.111111,2.0,1.0,2.000000,0.105263,0.052632,0.105263,0.000000,1.000000,0.111111


## Inspect run descriptor columns

In [8]:

run_cols = [c for c in df_run.columns if c.startswith("run_") and c not in {"run_length", "run_valid_residue_count"}]
len(run_cols), run_cols[:16]


(58,
 ['run_longest_homopolymer',
  'run_homopolymer_run_density',
  'run_repeated_block_fraction_len2',
  'run_repeated_block_fraction_len3',
  'run_charged_longest',
  'run_charged_n_runs',
  'run_charged_mean_run_length',
  'run_charged_longest_norm',
  'run_charged_run_density',
  'run_charged_fraction_in_runs_len2',
  'run_charged_fraction_in_runs_len3',
  'run_charged_blockiness',
  'run_charged_switching_freq',
  'run_hydrophobic_longest',
  'run_hydrophobic_n_runs',
  'run_hydrophobic_mean_run_length'])

In [9]:

df_run[
    [
        "sequence_id",
        "run_longest_homopolymer",
        "run_charged_longest",
        "run_hydrophobic_longest",
        "run_charged_n_runs",
        "run_hydrophobic_mean_run_length",
        "run_polar_switching_freq",
        "run_repeated_block_fraction_len2",
    ]
]


,sequence_id,run_longest_homopolymer,run_charged_longest,run_hydrophobic_longest,run_charged_n_runs,run_hydrophobic_mean_run_length,run_polar_switching_freq,run_repeated_block_fraction_len2
0,run_1,2.0,2.0,5.0,3.0,2.333333,0.478261,0.250000
1,run_2,15.0,0.0,0.0,0.0,NaN,0.000000,1.000000
2,run_3,4.0,18.0,0.0,1.0,NaN,0.000000,0.777778
3,run_4,1.0,2.0,3.0,4.0,1.800000,0.578947,0.000000
4,run_5,5.0,0.0,0.0,0.0,NaN,0.055556,0.947368
5,run_6,1.0,2.0,2.0,5.0,1.200000,0.700000,0.000000


## Dataset-level summary

In [10]:

run_summary = (
    df_run[run_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

run_summary.head(15)


,descriptor,mean_value
0,run_disorder_promoting_longest,8.166667
1,run_polar_mean_run_length,7.509524
2,run_polar_longest,7.000000
3,run_disorder_promoting_mean_run_length,5.627778
4,run_charged_mean_run_length,5.445833
5,run_longest_homopolymer,4.666667
6,run_charged_longest,4.000000
7,run_disorder_promoting_n_runs,3.500000
8,run_polar_n_runs,3.500000
9,run_order_promoting_n_runs,3.000000


## Sanity checks

In [11]:

assert "run_longest_homopolymer" in df_run.columns
assert "run_charged_longest" in df_run.columns
assert "run_hydrophobic_n_runs" in df_run.columns
assert "run_polar_mean_run_length" in df_run.columns
assert "run_aromatic_switching_freq" in df_run.columns
assert df_run["run_length"].min() > 0

print(f"Number of run/blockiness descriptor columns: {len(run_cols)}")
print("Run and blockiness descriptor checks passed.")


Number of run/blockiness descriptor columns: 58
Run and blockiness descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class RunBlockinessDescriptors:
    """Example class-style run/blockiness implementation for later migration into Roxy."""

    def transform_sequence(self, seq: str) -> dict:
        return run_blockiness_descriptors(seq)

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


run_transformer = RunBlockinessDescriptors()
run_matrix = run_transformer.transform(df_demo["sequence"].tolist())
run_matrix.head()


,run_length,run_valid_residue_count,run_longest_homopolymer,run_homopolymer_run_density,run_repeated_block_fraction_len2,run_repeated_block_fraction_len3,run_charged_longest,run_charged_n_runs,run_charged_mean_run_length,run_charged_longest_norm,...,run_disorder_promoting_switching_freq,run_order_promoting_longest,run_order_promoting_n_runs,run_order_promoting_mean_run_length,run_order_promoting_longest_norm,run_order_promoting_run_density,run_order_promoting_fraction_in_runs_len2,run_order_promoting_fraction_in_runs_len3,run_order_promoting_blockiness,run_order_promoting_switching_freq
0,24,24,2,0.875000,0.250000,0.000000,2,3,1.333333,0.083333,...,0.391304,5,5,2.400000,0.208333,0.208333,0.458333,0.208333,0.200000,0.434783
1,15,15,15,0.066667,1.000000,1.000000,0,0,NaN,0.000000,...,0.000000,0,0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
2,18,18,4,0.555556,0.777778,0.222222,18,1,18.000000,1.000000,...,0.117647,0,0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
3,20,20,1,1.000000,0.000000,0.000000,2,4,1.250000,0.100000,...,0.473684,3,6,1.333333,0.150000,0.300000,0.150000,0.150000,0.166667,0.578947
4,19,19,5,0.315789,0.947368,0.842105,0,0,NaN,0.000000,...,0.111111,2,1,2.000000,0.105263,0.052632,0.105263,0.000000,1.000000,0.111111


## Merge transformer output back to the dataset

In [13]:

df_run_class = pd.concat([df_demo, run_matrix], axis=1)
df_run_class.head()


,sequence_id,sequence,label,run_length,run_valid_residue_count,run_longest_homopolymer,run_homopolymer_run_density,run_repeated_block_fraction_len2,run_repeated_block_fraction_len3,run_charged_longest,...,run_disorder_promoting_switching_freq,run_order_promoting_longest,run_order_promoting_n_runs,run_order_promoting_mean_run_length,run_order_promoting_longest_norm,run_order_promoting_run_density,run_order_promoting_fraction_in_runs_len2,run_order_promoting_fraction_in_runs_len3,run_order_promoting_blockiness,run_order_promoting_switching_freq
0,run_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,2,0.875000,0.250000,0.000000,2,...,0.391304,5,5,2.400000,0.208333,0.208333,0.458333,0.208333,0.200000,0.434783
1,run_2,GGGGGGGGGGGGGGG,B,15,15,15,0.066667,1.000000,1.000000,0,...,0.000000,0,0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
2,run_3,KRRKRRKRRKRRDDDDEE,A,18,18,4,0.555556,0.777778,0.222222,18,...,0.117647,0,0,NaN,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
3,run_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,1,1.000000,0.000000,0.000000,2,...,0.473684,3,6,1.333333,0.150000,0.300000,0.150000,0.150000,0.166667,0.578947
4,run_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,5,0.315789,0.947368,0.842105,0,...,0.111111,2,1,2.000000,0.105263,0.052632,0.105263,0.000000,1.000000,0.111111



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move the helper logic into `roxy/sequence/complexity.py` or a dedicated `runs.py`
- keep residue groups in `roxy/core/constants.py`
- expose a class such as `RunBlockinessDescriptors`
- allow configurable:
  - tracked groups
  - run length thresholds
  - which run/blockiness summaries to compute
- add tests for:
  - empty sequences
  - homopolymer-rich sequences
  - highly alternating sequences
  - long hydrophobic blocks
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [ ]:
# df_run.to_csv("demo_run_blockiness_descriptors.csv", index=False)
